# Part 10, notebook 2 of 5: a supervisor and two specialists

Companion to *When One Agent Is Not Enough*. The five notebooks follow the post in order. Each explains the idea with a daily-life picture, shows the real source, and runs it. **Cells marked "Your turn" are for you to change and rerun.**

| # | Notebook | Post section | Needs the model server? |
|---|---|---|---|
| 10a | `10a_three_ways` | Three ways to spend model contexts | two short live cells |
| 10b | `10b_supervisor_and_handoff` | A supervisor and two specialists | live cells (about 1 minute) |
| 10c | `10c_fanout_and_panel` | One sub-agent per box, and a panel | live cells (about 30 seconds) |
| 10d | `10d_the_200_image_results` | The results | no (reads the saved runs) |
| 10e | `10e_failures_and_tests` | What goes wrong | no |

**The picture.** The manager at the clinic's front desk cannot see the patient. The nurse measures and writes a note. The manager copies what
matters onto a form for the doctor, and the doctor, who has never met the patient, decides from the form alone. Whatever is not on the form does not exist for the doctor.

That is the **hand-off**: the only thing that crosses from one room to another is text somebody wrote. The question for this notebook is how much of the truth survives it.

In [1]:
import sys, json, os
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))     # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
os.environ.setdefault("PART10_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves
import sub_agents as sa
import sub_summary as ss
%matplotlib inline

## One saved supervisor run

This is a real record from the 200-image run (nothing here calls the model). `note` is what the measuring specialist told the supervisor,
`brief` is what the supervisor wrote for the judge, `judge_out` is the judge's answer.

In [2]:
d = ss.load()
sup = d["supervisor"][0]
r = sup["test_00046"]
print("tools the supervisor called:", r["tools"])
print("\n--- brief the supervisor wrote for the judge ---\n" + r["brief"])
print("\n--- the judge's answer ---\n" + r["judge_out"])
print("\nkept:", r["kept"], "| calls by role:", {k: v["calls"] for k, v in r["by_role"].items()})

tools the supervisor called: ['inspect_stage', 'judge_stage', 'submit_report']

--- brief the supervisor wrote for the judge ---
Image test_00046, 4 candidate boxes measured. All are brighter than surroundings.
Cand 0: box [83,71,104,71,94], contrast +30.29 (brighter), area 483 px, aspect 0.91, edge dist 71 px (well inside image).
Cand 1: box [141,153,219,224], contrast +8.96, area 60 px, aspect 2.4, edge dist 0 (touches bottom edge).
Cand 2: box [85,90,221,224], contrast +17.67, area 15 px, aspect 1.67, edge dist 0 (touches bottom edge).
Cand 3: box [91,95,221,224], contrast +15.91, area 12 px, aspect 1.33, edge dist 0 (touches bottom edge).
Which candidates are real defects?

--- the judge's answer ---
Cand 0: defect 0.7
Cand 1: false_alarm 0.9
Cand 2: false_alarm 0.8
Cand 3: false_alarm 0.8

kept: [0] | calls by role: {'supervisor': 3, 'measurer': 3, 'judge': 1}


## How much survived the hand-off? (all 200 saved runs)

`fidelity` checks, for every candidate box, that its id and its **contrast number** appear in the brief (rounding to one decimal is forgiven, rounding to a whole number is not).
The second line checks the **box coordinates**, which the judge never needs.

In [3]:
s = ss.summarize()
h = s["handoff"]
print(f"images where the supervisor wrote a brief: {h['images_with_brief']}  (the other {h['images_no_judge_call']} had no candidates)")
print(f"contrast carried:    {h['carried']} of {h['candidates']} boxes; every number intact in {h['images_complete']} of {h['images_with_brief']} briefs")
bc = h["box_coordinates"]
print(f"coordinates carried: {bc['carried']} of {bc['boxes']} boxes")
print(f"supervisor kept a box the judge had not called a defect: {h['followed_judge']['images'] - h['followed_judge']['kept_subset_of_judge_defects']} times")

images where the supervisor wrote a brief: 145  (the other 55 had no candidates)
contrast carried:    512 of 513 boxes; every number intact in 144 of 145 briefs
coordinates carried: 386 of 513 boxes
supervisor kept a box the judge had not called a defect: 0 times


## Your turn: be the supervisor

Write the brief yourself and let the real judge sub-agent read it. Try a vague brief, a brief with the numbers rounded, a brief that leaves a box out.
`fidelity` tells you how much of the truth you carried. The judge call goes to the model server.

In [4]:
IMAGE = "test_00046"
MY_BRIEF = """Image test_00046. Candidate 0 is bright and big. Candidate 1, 2 and 3 are small and touch the edge."""   # edit me

meter = sa.Meter()
msg = sa.chat([{"role": "system", "content": sa.SYSTEM_JUDGE}, {"role": "user", "content": MY_BRIEF}], meter, "judge")
print("the judge says:\n" + (msg.content or "").strip())
print("\ncarried:", sa.fidelity(IMAGE, MY_BRIEF), "(boxes in the image / boxes whose id and contrast are in your brief)")
print("true measurements:", [(m["candidate_id"], m["contrast"], m["area_px"], m["edge_distance_px"]) for m in sa.measure_all(IMAGE)])

the judge says:
0 defect 0.8
1 false_alarm 0.9
2 false_alarm 0.9
3 false_alarm 0.9

carried: {'candidates': 4, 'carried': 0} (boxes in the image / boxes whose id and contrast are in your brief)
true measurements: [(0, 30.29, 483, 71), (1, 8.96, 60, 0), (2, 17.67, 15, 0), (3, 15.91, 12, 0)]


## And live, the whole supervisor on one image

One image costs about 6 model calls and 40 seconds here, against 3 calls for the single agent.

In [5]:
r = sa.arm_supervisor("test_00002")
print("tools:", r["tools"])
print("calls by role:", {k: (v["calls"], v["prompt"]) for k, v in r["by_role"].items()}, " (calls, prompt tokens)")
print("kept:", r["kept"], "| tp/fp/fn:", r["tp"], r["fp"], r["fn"], "| hand-off:", r["handoff"])

tools: ['inspect_stage', 'judge_stage', 'submit_report']
calls by role: {'supervisor': (3, 2637), 'measurer': (3, 2356), 'judge': (1, 426)}  (calls, prompt tokens)
kept: [] | tp/fp/fn: 0 0 1 | hand-off: {'candidates': 3, 'carried': 3}
